# 🏗️ Agentic Document Processing — Azure Solution

**Built for:** Rapid Circle Technical Interview  
**Stack:** Azure OpenAI · Azure Document Intelligence · Azure Service Bus · Azure Container Apps · Azure Key Vault · Cosmos DB

---

## Azure Architecture

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                        LAYER 1 — INGESTION                                  │
│                                                                             │
│  Documents ──► Azure Blob Storage ──► Azure Event Grid ──► Azure Service Bus│
│  (PDF/email/    (raw document store)   (upload trigger)    (durable queue)  │
│   invoice)                                                                  │
└─────────────────────────────────┬───────────────────────────────────────────┘
                                  │ dequeue
┌─────────────────────────────────▼───────────────────────────────────────────┐
│                     LAYER 2 — AI AGENT ORCHESTRATION                        │
│                     (Azure Container Apps — scales 0→20)                    │
│                                                                             │
│  ┌─────────────────┐  ┌──────────────────┐  ┌────────────┐  ┌───────────┐  │
│  │ Classifier Agent│─►│ Extractor Agent  │─►│ Validator  │─►│  Router   │  │
│  │ Azure OpenAI    │  │ Azure Doc Intel. │  │ Agent      │  │  Agent    │  │
│  │ GPT-4o          │  │ Form Recognizer  │  │ Biz Rules  │  │  Config   │  │
│  │ confidence score│  │ structured fields│  │ validation │  │  driven   │  │
│  └─────────────────┘  └──────────────────┘  └────────────┘  └───────────┘  │
│           │ low confidence / validation fail              │ routed           │
└───────────┼───────────────────────────────────────────────┼─────────────────┘
            │                                               │
┌───────────▼───────────┐          ┌────────────────────────▼─────────────────┐
│  LAYER 3 — HITL       │          │  LAYER 4 — DOWNSTREAM SYSTEMS            │
│                       │          │                                          │
│  Service Bus          │          │  Finance API   CRM API   Compliance API  │
│  (review queue)       │          │  (SAP/D365)  (Salesforce) (Audit Reg.)  │
│  Power Apps Portal    │          │                                          │
│  SLA tracking         │─────────►│  (re-inject after human correction)      │
└───────────────────────┘          └──────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────┐
│                     LAYER 5 — OBSERVABILITY & SECURITY                      │
│                                                                             │
│  Azure Monitor · App Insights · Cosmos DB (audit) · Key Vault · Defender   │
└─────────────────────────────────────────────────────────────────────────────┘
```

---

## Azure Services Used

| Layer | Azure Service | Purpose |
|---|---|---|
| Ingestion | **Azure Blob Storage** | Raw document store — PDFs, emails, invoices |
| Ingestion | **Azure Event Grid** | Fires trigger on every blob upload |
| Ingestion | **Azure Service Bus** | Durable queue — dead-letter, retry, no lost messages |
| AI Layer | **Azure OpenAI (GPT-4o)** | Document classification + confidence scoring |
| AI Layer | **Azure Document Intelligence** | Structured field extraction from PDFs |
| Compute | **Azure Container Apps** | Serverless — scales 0→20 based on queue depth |
| HITL | **Azure Service Bus (review queue)** | Escalated documents queue with SLA |
| HITL | **Power Apps Portal** | Human review UI — no code required |
| Audit | **Azure Cosmos DB** | Immutable audit trail — queryable, 3-year retention |
| Security | **Azure Key Vault** | All secrets — no passwords in code |
| Security | **Managed Identity** | Identity-based access — no API keys in env vars |
| Security | **Azure RBAC** | Finance staff see only Finance documents |
| Observability | **Azure Application Insights** | Distributed tracing, metrics, alerting |
| Observability | **Azure Monitor** | Dashboards — confidence scores, SLA breach, throughput |
| Threat | **Microsoft Defender for Cloud** | Threat detection on storage and compute |


---
## Setup — Run these two cells first every session

In [ ]:
# Cell 1 — Set working directory and Python path
import os, sys

os.chdir(os.path.expanduser('~/Downloads/doc-agent'))
sys.path.insert(0, os.getcwd())

print('✅ Working directory:', os.getcwd())
print('✅ Python:', sys.executable)

In [ ]:
# Cell 2 — Start the Azure-ready API server
# In production: this runs in Azure Container Apps
# Locally: same code, LOCAL_DEV=true uses stubs instead of real Azure services

import subprocess, time, requests

server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'main:app',
     '--port', '8000', '--log-level', 'warning'],
    cwd=os.getcwd(),
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print('Starting server (Azure Container Apps equivalent)...')
for i in range(15):
    time.sleep(1)
    try:
        r = requests.get('http://localhost:8000/health', timeout=2)
        if r.status_code == 200:
            print(f'✅ Server ready after {i+1}s')
            print(f'   Local URL  : http://localhost:8000')
            print(f'   Azure URL  : https://<app>.azurecontainerapps.io (after deployment)')
            print(f'   Swagger UI : http://localhost:8000/docs')
            break
    except:
        print(f'  waiting... ({i+1}s)')
else:
    print('❌ Server failed:', server.stderr.read().decode())

---
## Demo 1 — Invoice → Azure Finance API (Happy Path)

In [ ]:
# Simulates: document uploaded to Azure Blob Storage
# Azure Event Grid fires → Service Bus queues it → Container App processes it
# Classifier uses Azure OpenAI GPT-4o (stub locally)
# Extractor uses Azure Document Intelligence (stub locally)

import json, requests

resp = requests.post('http://localhost:8000/documents/text', json={
    'filename': 'invoice.txt',
    'text_content': '''INVOICE
Vendor: Acme Supplies Ltd
Invoice No: INV-2024-001
Total Amount Due: $8,250.00
Due Date: 2024-02-15
PO Number: PO-9876'''
})

data = resp.json()
print('=== AZURE PIPELINE RESULT ===')
print(f"Document ID    : {data['id']}")
print(f"Classified as  : {data['doc_type']} (Azure OpenAI GPT-4o)")
print(f"Confidence     : {data['confidence']} (threshold: 0.80)")
print(f"Extracted by   : Azure Document Intelligence")
print(f"Fields found   : {list(data['extracted_fields'].keys())}")
print(f"Validation     : {'✅ Passed' if data['validation_passed'] else '❌ Failed'}")
print(f"Routed to      : {data['destination']} API")
print(f"Final status   : {data['status']}")
print(f"\nAudit trail ({len(data['audit_trail'])} entries — stored in Azure Cosmos DB):")
for entry in data['audit_trail']:
    print(f"  [{entry['timestamp'][11:19]}] {entry['agent']:12} → {entry['action']}")

---
## Demo 2 — High Value Invoice → Azure HITL Escalation

In [ ]:
# Validator catches $75,000 > $50,000 threshold
# In Azure: escalated message goes to Service Bus 'human-review' queue
# Power Apps portal picks it up and shows it to the reviewer

resp = requests.post('http://localhost:8000/documents/text', json={
    'filename': 'big_invoice.txt',
    'text_content': '''INVOICE
Vendor: BigCorp International
Invoice No: INV-999
Total Amount Due: $75,000.00
Due Date: 2024-03-01'''
})

data = resp.json()
doc_id = data['id']

print('=== AZURE ESCALATION TRIGGERED ===')
print(f"Status         : {data['status']}")
print(f"Reason         : {data['escalation_reason']}")
print(f"Note           : {data['escalation_note']}")
print(f"Doc ID         : {doc_id}")
print()
print('In Azure production:')
print('  → Message pushed to Service Bus human-review queue')
print('  → Power Apps portal shows document to reviewer')
print('  → SLA timer starts (4 hours)')
print('  → Azure Monitor alert fires if SLA breached')

---
## Demo 3 — Human Review Queue (Power Apps equivalent)

In [ ]:
# In Azure: this data feeds the Power Apps review portal
# Reviewers see: document, AI's best guess, confidence score, SLA deadline

resp = requests.get('http://localhost:8000/review/')
pending = resp.json()

print('=== AZURE HUMAN REVIEW QUEUE ===')
print(f"{len(pending)} document(s) awaiting review\n")
for item in pending:
    print(f"  File          : {item['filename']}")
    print(f"  AI guess      : {item['doc_type']} ({item['confidence']:.0%} confident)")
    print(f"  Escalation    : {item['escalation_reason']}")
    print(f"  SLA deadline  : {item['sla_deadline']}")
    print(f"  SLA breached  : {item['sla_breached']}")
    print(f"  Errors        : {item['validation_errors']}")

---
## Demo 4 — Human Resolves → Re-injected into Pipeline

In [ ]:
# Human corrects the document in Power Apps portal
# Document re-enters pipeline at Validator (skips Classifier — human confirmed type)
# Azure Service Bus: message moves from review queue back to processing

resp = requests.post(f'http://localhost:8000/review/{doc_id}/resolve', json={
    'reviewer': 'jane.smith@rapidcircle.com',
    'doc_type': 'invoice',
    'extracted_fields': {
        'vendor_name': 'BigCorp International',
        'invoice_number': 'INV-999',
        'amount': 8000.00,          # corrected — below $50k threshold
        'due_date': '2024-03-01',
        'po_number': 'PO-APPROVED-001'
    }
})

final = resp.json()
print('=== HUMAN REVIEW COMPLETE ===')
print(f"Reviewer       : jane.smith@rapidcircle.com")
print(f"Correction     : Amount changed to $8,000 (below $50k threshold)")
print(f"Final status   : {final['status']}")
print(f"Routed to      : {final['destination']} API")
print(f"Confidence     : {final['confidence']} (1.0 = human confirmed)")
print()
print('Audit trail shows full chain:')
for entry in final['audit_trail']:
    print(f"  [{entry['timestamp'][11:19]}] {entry['agent']:12} → {entry['action']}")

---
## Demo 5 — All 4 Document Types

In [ ]:
# Send all 4 document types and show routing

documents = [
    {
        'filename': 'contract.txt',
        'text_content': '''SERVICE AGREEMENT (NDA)
This Agreement is made between Rapid Circle BV (Party A)
and TechPartner Ltd (Party B), effective date: 2024-01-01.
Signed by: Jane Smith, Director'''
    },
    {
        'filename': 'email.txt',
        'text_content': '''From: customer@example.com
To: support@rapidcircle.com
Subject: Upgrade my subscription
Date: Mon, 15 Jan 2024 09:30:00

Hi, I would like to upgrade my plan. Please advise.'''
    },
    {
        'filename': 'compliance.txt',
        'text_content': '''GDPR Compliance Filing
Regulation: GDPR-2018/EU
Entity: Rapid Circle BV
Submission Date: 2024-01-31
Jurisdiction: European Union
Responsible Officer: Maria Compliance'''
    }
]

print('=== ALL DOCUMENT TYPES — AZURE ROUTING ===')
print(f"{'File':<20} {'Type':<12} {'Confidence':<12} {'Destination':<15} {'Status'}")
print('-' * 75)

for doc in documents:
    r = requests.post('http://localhost:8000/documents/text', json=doc)
    d = r.json()
    status = '✅ routed' if d['status'] == 'routed' else '⚠️  ' + d['status']
    print(f"{d['filename']:<20} {d['doc_type']:<12} {d['confidence']:<12.2f} {str(d.get('destination','—')):<15} {status}")

---
## Demo 6 — Azure Monitor Metrics Dashboard

In [ ]:
# In Azure: this feeds Azure Monitor Workbook dashboard
# Metrics shipped to App Insights via OpenTelemetry

resp = requests.get('http://localhost:8000/metrics')
m = resp.json()

print('=== AZURE MONITOR METRICS ===')
print(f"Total processed    : {m['total_documents']}")
print(f"Pending reviews    : {m['pending_reviews']}")
print()
print('By status (Azure Monitor chart):') 
for status, count in m['by_status'].items():
    bar = '█' * count
    print(f"  {status:<20} {bar} ({count})")
print()
print('By document type:')
for doc_type, count in m['by_doc_type'].items():
    bar = '█' * count
    print(f"  {doc_type:<20} {bar} ({count})")
print()
print('Escalation reasons (triggers Azure Monitor alert):')
for reason, count in m.get('escalation_reasons', {}).items():
    print(f"  {reason:<30} {count}")
print()
print('In Azure production, these metrics feed:')
print('  → Azure Monitor Workbook (live dashboard)')
print('  → App Insights (distributed traces per document)')
print('  → Azure Alerts (SLA breach, high escalation rate)')

---
## Demo 7 — Extensibility: Add a New Document Type

In [ ]:
# Demonstrates the key interview point:
# Adding a new document type = edit JSON config, no code changes
# In Azure: update config in App Configuration service, redeploy

import json

print('=== HOW TO ADD A NEW DOCUMENT TYPE ===')
print()
print('Step 1: Add to config/extraction_schemas.json')
print(json.dumps({
    'purchase_order': {
        'required_fields': ['supplier', 'po_number', 'delivery_date', 'total_value'],
        'optional_fields': ['line_items', 'payment_terms'],
        'validation_rules': {'require_approval_above': 25000}
    }
}, indent=2))
print()
print('Step 2: Add to config/routing_rules.json')
print(json.dumps({
    'doc_type': 'purchase_order',
    'destination': 'finance',
    'description': 'Purchase orders go to Finance for approval'
}, indent=2))
print()
print('Step 3: Add to DocumentType enum in config/models.py')
print('  PURCHASE_ORDER = "purchase_order"')
print()
print('That is ALL. No agent code changes. No pipeline changes.')
print('In Azure: update Azure App Configuration, Container App restarts automatically.')

---
## Azure Deployment — What this looks like in production

In [ ]:
# Show the Azure deployment commands
# These would actually deploy this exact code to Azure

print('=== AZURE DEPLOYMENT COMMANDS ===')
print()
print('# 1. Login to Azure')
print('az login')
print()
print('# 2. Create Resource Group')
print('az group create --name rg-doc-agent --location australiaeast')
print()
print('# 3. Deploy all Azure infrastructure (Bicep template)')
print('az deployment group create \\')
print('  --resource-group rg-doc-agent \\')
print('  --template-file azure/main.bicep \\')
print('  --parameters @azure/parameters.json')
print()
print('# 4. Build and push Docker image to Azure Container Registry')
print('az acr login --name docagentregistry')
print('docker build -t doc-agent:v1 .')
print('docker tag doc-agent:v1 docagentregistry.azurecr.io/doc-agent:v1')
print('docker push docagentregistry.azurecr.io/doc-agent:v1')
print()
print('# 5. Store secrets in Key Vault (no secrets in code)')
print('az keyvault secret set --vault-name docagent-kv --name openai-key --value <KEY>')
print()
print('# Result: Live Azure URL')
print('# https://docagent-api.<region>.azurecontainerapps.io/docs')
print()
print('# Deployment script: azure/deploy.sh')
print('# Infrastructure template: azure/main.bicep')

In [ ]:
# Stop server when done
server.terminate()
print('✅ Server stopped')